In [ ]:
!pip install transformers datasets evaluate sacrebleu torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.6 MB/s eta 0:00:00


In [9]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import evaluate
import numpy as np
import warnings

# Ignore warning messages for cleaner output
warnings.filterwarnings("ignore")

# 1. Load the English-Vietnamese dataset
# We use IWSLT 2015, a standard benchmark dataset for translation
print("Loading dataset...")
raw_datasets = load_dataset("Helsinki-NLP/opus-100", "en-vi")

# 2. Preprocessing and Tokenization
print("Initializing tokenizer...")
model_checkpoint = "Helsinki-NLP/opus-mt-en-vi"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    # Extract source (English) and target (Vietnamese) sentences
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["vi"] for ex in examples["translation"]]

    # Tokenize both inputs and targets using the new syntax
    model_inputs = tokenizer(inputs, text_target=targets, max_length=128, truncation=True)

    return model_inputs

print("Tokenizing data...")
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

# 3. Model Initialization and Evaluation Setup
print("Loading pre-trained model...")
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

# 4. Training Configuration
# Set up arguments for fine-tuning
training_args = Seq2SeqTrainingArguments(
    output_dir="./mt-en-vi-model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True, # Enable mixed precision for faster GPU training on Colab
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    # Using a subset of 1000 sentences for quick demonstration
    # Remove '.select(range(1000))' to train on the full dataset later
    train_dataset=tokenized_datasets["train"].select(range(1000)),
    eval_dataset=tokenized_datasets["validation"].select(range(200)),
    data_collator=data_collator,
   processing_class=tokenizer,
    compute_metrics=compute_metrics
)

# Start training
print("Starting model training (fine-tuning)...")
trainer.train()
print("Training completed!")

# 5. Inference and Error Analysis Function
def translate_sentence(text):
    inputs = tokenizer(text, return_tensors="pt").input_ids
    inputs = inputs.to(model.device) # Move to GPU if available

    outputs = model.generate(inputs, max_length=128, num_beams=4, early_stopping=True)
    translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated_text

# Test with various sentence structures for the Error Analysis report section
test_sentences = [
    "The student is working on a natural language processing project.", # Standard sentence
    "It's raining cats and dogs out there.", # Idiom (Likely to be translated literally)
    "He kicked the bucket yesterday.", # Slang/Idiom
    "I'm feeling a bit under the weather today." # Metaphor
]

print("\n--- Translation Test Results ---")
for sentence in test_sentences:
    vietnamese_translation = translate_sentence(sentence)
    print(f"Source: {sentence}")
    print(f"Translation: {vietnamese_translation}\n")

Loading dataset...
Initializing tokenizer...
Tokenizing data...
Loading pre-trained model...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting model training (fine-tuning)...


Epoch,Training Loss,Validation Loss,Bleu
1,No log,1.758178,23.262042
2,No log,1.735702,23.289276
3,No log,1.732332,23.900087


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training completed!

--- Translation Test Results ---
Source: The student is working on a natural language processing project.
Translation: Sinh viên đang nghiên cứu một dự án xử lý ngôn ngữ tự nhiên.

Source: It's raining cats and dogs out there.
Translation: Ngoài kia toàn chó và mèo mưa.

Source: He kicked the bucket yesterday.
Translation: Hôm qua nó đã đá cái xô.

Source: I'm feeling a bit under the weather today.
Translation: Hôm nay tôi cảm thấy hơi mệt.

